In [1]:
import pandas as pd
import os
import re

AA_STAB = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint9_doubleml_stability_results.csv"
EA_STAB = r"C:\Users\user\Downloads\GSE148812_clean\checkpoint9_doubleml_stability_results_relatedness_filtered.csv"
OUT_DIR = r"C:\Users\user\Desktop\ai causal\causal_project\cross_ancestry\smoking_status_corrected"
os.makedirs(OUT_DIR, exist_ok=True)

def strip_suffix(pid):
    return re.sub(r'_\d+$', '', pid)

aa_stab = pd.read_csv(AA_STAB)
ea_stab = pd.read_csv(EA_STAB)

aa_stab["core_name"] = aa_stab["probe_id"].map(strip_suffix)
ea_stab["core_name"] = ea_stab["probe_id"].map(strip_suffix)

print("AA SNPs (corrected):", len(aa_stab))
print("EA SNPs (corrected):", len(ea_stab))

merged = aa_stab.merge(ea_stab, on="core_name", how="inner", suffixes=("_AA", "_EA"))
print("Merged (present in both):", len(merged))

merged.to_csv(os.path.join(OUT_DIR, "merged_stability_AA_EA_smoking_corrected.csv"), index=False)
print("Saved.")

AA SNPs (corrected): 141324
EA SNPs (corrected): 127416
Merged (present in both): 103048
Saved.


In [2]:
import pandas as pd
import os

OUT_DIR = r"C:\Users\user\Desktop\ai causal\causal_project\cross_ancestry\smoking_status_corrected"
merged = pd.read_csv(os.path.join(OUT_DIR, "merged_stability_AA_EA_smoking_corrected.csv"))

print("=== Joint stability counts (corrected pipeline) ===")
for thresh in [0.5, 0.7, 0.8, 0.9, 1.0]:
    aa_stable = merged["stability_fraction_AA"] >= thresh
    ea_stable = merged["stability_fraction_EA"] >= thresh
    both = (aa_stable & ea_stable).sum()
    aa_only = (aa_stable & ~ea_stable).sum()
    ea_only = (~aa_stable & ea_stable).sum()
    print(f">= {thresh:.0%}  |  AA: {aa_stable.sum()}  EA: {ea_stable.sum()}  "
          f"BOTH: {both}  AA-only: {aa_only}  EA-only: {ea_only}")

corr = merged["stability_fraction_AA"].corr(merged["stability_fraction_EA"])
print(f"\nGenome-wide correlation: {corr:.4f}")

# specifically check PPP1R12B's cross-ancestry stability now
ppp1r12b_check = merged[merged["core_name"].str.contains("2277017", na=False)]
if len(ppp1r12b_check) > 0:
    print("\n=== PPP1R12B cross-ancestry stability (corrected) ===")
    print(ppp1r12b_check[["core_name", "stability_fraction_AA", "stability_fraction_EA"]])

=== Joint stability counts (corrected pipeline) ===
>= 50%  |  AA: 1447  EA: 1604  BOTH: 6  AA-only: 1441  EA-only: 1598
>= 70%  |  AA: 600  EA: 651  BOTH: 2  AA-only: 598  EA-only: 649
>= 80%  |  AA: 369  EA: 341  BOTH: 1  AA-only: 368  EA-only: 340
>= 90%  |  AA: 67  EA: 95  BOTH: 0  AA-only: 67  EA-only: 95
>= 100%  |  AA: 8  EA: 22  BOTH: 0  AA-only: 8  EA-only: 22

Genome-wide correlation: -0.0553

=== PPP1R12B cross-ancestry stability (corrected) ===
          core_name  stability_fraction_AA  stability_fraction_EA
0  exm2277017-0_T_R                    1.0               0.066667


In [3]:
overlap_80 = merged[(merged["stability_fraction_AA"] >= 0.8) & (merged["stability_fraction_EA"] >= 0.8)]
print(overlap_80[["core_name", "stability_fraction_AA", "stability_fraction_EA"]])

          core_name  stability_fraction_AA  stability_fraction_EA
217  exm72322-0_B_R               0.833333                    0.8


In [4]:
import pandas as pd
import re
import requests
import time

def strip_address_suffix(pid):
    return re.sub(r'_\d+$', '', pid)

def query_ensembl_grch37(chrom, pos, max_retries=3):
    url = f"https://grch37.rest.ensembl.org/overlap/region/human/{chrom}:{int(pos)-1}-{int(pos)+1}?feature=gene;content-type=application/json"
    headers = {"User-Agent": "Mozilla/5.0 (research script)"}
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, headers=headers, timeout=15)
            if resp.status_code == 200:
                return resp.json()
            time.sleep(2 * (attempt + 1))
        except Exception:
            time.sleep(2 * (attempt + 1))
    return None

manifest_path = r"C:\Users\user\Downloads\HumanExome-12-v1-0-B.csv"
manifest_df = pd.read_csv(manifest_path, skiprows=7, low_memory=False)
manifest_df["core_name"] = manifest_df["IlmnID"].map(strip_address_suffix)
pos_lookup = manifest_df.set_index("core_name")[["Chr", "MapInfo"]]

core = "exm72322-0_B_R"
if core in pos_lookup.index:
    chrom = pos_lookup.loc[core, "Chr"]
    pos = pos_lookup.loc[core, "MapInfo"]
    print(f"Position: chr{chrom}:{pos}")
    data = query_ensembl_grch37(chrom, pos)
    gene = data[0].get("external_name", "UNKNOWN") if data else f"intergenic_chr{chrom}"
    print(f"Gene: {gene}")
else:
    print("Not found in manifest")

Position: chr1:86907149.0
Gene: CLCA2


In [5]:
import pandas as pd
import os

OUT_DIR = r"C:\Users\user\Desktop\ai causal\causal_project\cross_ancestry\smoking_status_corrected"
merged = pd.read_csv(os.path.join(OUT_DIR, "merged_stability_AA_EA_smoking_corrected.csv"))

overlap_70 = merged[(merged["stability_fraction_AA"] >= 0.7) & (merged["stability_fraction_EA"] >= 0.7)]
print(f"SNPs overlapping at >=70% in both ancestries: {len(overlap_70)}")
print(overlap_70[["core_name", "stability_fraction_AA", "stability_fraction_EA"]])

SNPs overlapping at >=70% in both ancestries: 2
            core_name  stability_fraction_AA  stability_fraction_EA
217    exm72322-0_B_R               0.833333                    0.8
422  exm1192695-0_B_R               0.733333                    0.8


In [6]:
import pandas as pd
import re
import requests
import time

def strip_address_suffix(pid):
    return re.sub(r'_\d+$', '', pid)

def query_ensembl_grch37(chrom, pos, max_retries=3):
    url = f"https://grch37.rest.ensembl.org/overlap/region/human/{chrom}:{int(pos)-1}-{int(pos)+1}?feature=gene;content-type=application/json"
    headers = {"User-Agent": "Mozilla/5.0 (research script)"}
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, headers=headers, timeout=15)
            if resp.status_code == 200:
                return resp.json()
            time.sleep(2 * (attempt + 1))
        except Exception:
            time.sleep(2 * (attempt + 1))
    return None

manifest_path = r"C:\Users\user\Downloads\HumanExome-12-v1-0-B.csv"
manifest_df = pd.read_csv(manifest_path, skiprows=7, low_memory=False)
manifest_df["core_name"] = manifest_df["IlmnID"].map(strip_address_suffix)
pos_lookup = manifest_df.set_index("core_name")[["Chr", "MapInfo"]]

core = "exm1192695-0_B_R"
if core in pos_lookup.index:
    chrom = pos_lookup.loc[core, "Chr"]
    pos = pos_lookup.loc[core, "MapInfo"]
    print(f"Position: chr{chrom}:{pos}")
    data = query_ensembl_grch37(chrom, pos)
    gene = data[0].get("external_name", "UNKNOWN") if data else f"intergenic_chr{chrom}"
    print(f"Gene: {gene}")
else:
    print("Not found in manifest")

Position: chr15:99679573.0
Gene: RP11-6O2.3
